# Module 4: Water Body Size Classification

**Task 4**: Classify water bodies by size classes (<1ha, 1–50ha, 50–100ha, 100–200ha, 200–300ha, >300ha) — State level analysis for three periods.

**Deliverables**:
- Water body polygon files
- Size classification summary tables (count + area per class per year)
- Grouped bar charts
- Spatial maps colored by size class

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from config import (
    STATES, YEARS, LULC_CLASSES,
    GEE_EXPORTS_DIR, PROCESSED_DIR, FIGURES_DIR,
    WATER_SIZE_CLASSES, BOUNDARIES_DIR,
)
from scripts.visualization import save_figure, plot_water_size_distribution
from scripts.lulc_utils import save_composition_csv

print('Imports successful!')

## 4.1: Load Vectorized Water Body Polygons

These were exported from GEE at 10m native resolution in Module 1.

In [ ]:
water_bodies = {}  # {state_key: {year: GeoDataFrame}}

for state_key, state_cfg in STATES.items():
    water_bodies[state_key] = {}
    
    for year in YEARS:
        filepath = GEE_EXPORTS_DIR / f'{state_key}_water_bodies_{year}.geojson'
        
        if filepath.exists():
            gdf = gpd.read_file(filepath)
            
            # Ensure area_ha column exists
            if 'area_ha' not in gdf.columns:
                # Reproject to UTM for accurate area calc
                utm_epsg = state_cfg['utm_epsg']
                gdf_utm = gdf.to_crs(epsg=utm_epsg)
                gdf['area_ha'] = gdf_utm.geometry.area / 10000  # m² → ha
            
            water_bodies[state_key][year] = gdf
            print(f'{state_cfg["name"]} {year}: {len(gdf)} water bodies loaded')
        else:
            print(f'⚠️ Not found: {filepath.name}')

## 4.2: Classify Water Bodies by Size

In [ ]:
def classify_water_size(area_ha):
    """Assign size class based on area in hectares."""
    for sc in WATER_SIZE_CLASSES:
        if sc['min_ha'] <= area_ha < sc['max_ha']:
            return sc['label']
    return WATER_SIZE_CLASSES[-1]['label']  # Mega


# Classify all water bodies
for state_key in water_bodies:
    for year in water_bodies[state_key]:
        gdf = water_bodies[state_key][year]
        gdf['size_class'] = gdf['area_ha'].apply(classify_water_size)
        
        # Save classified water bodies
        outdir = PROCESSED_DIR / 'water_bodies'
        outdir.mkdir(parents=True, exist_ok=True)
        gdf.to_file(outdir / f'{state_key}_water_classified_{year}.geojson', driver='GeoJSON')
        
        print(f'{state_key} {year}: classified {len(gdf)} water bodies')

## 4.3: State-Level Summary Tables

In [ ]:
size_labels = [sc['label'] for sc in WATER_SIZE_CLASSES]

all_size_tables = {}

for state_key, state_cfg in STATES.items():
    rows = []
    
    for year in YEARS:
        if year not in water_bodies.get(state_key, {}):
            continue
        
        gdf = water_bodies[state_key][year]
        
        for sc_label in size_labels:
            subset = gdf[gdf['size_class'] == sc_label]
            rows.append({
                'year': year,
                'size_class': sc_label,
                'count': len(subset),
                'total_area_ha': subset['area_ha'].sum(),
                'mean_area_ha': subset['area_ha'].mean() if len(subset) > 0 else 0,
            })
    
    size_df = pd.DataFrame(rows)
    all_size_tables[state_key] = size_df
    
    # Save
    save_composition_csv(
        size_df,
        PROCESSED_DIR / 'water_bodies' / f'{state_key}_water_size_summary.csv'
    )
    
    print(f'\n{state_cfg["name"]} — Water Body Summary:')
    pivot = size_df.pivot_table(index='size_class', columns='year',
                                values=['count', 'total_area_ha'], fill_value=0)
    display(pivot)

## 4.4: Change Analysis

In [ ]:
# Analyze changes between periods
for state_key, state_cfg in STATES.items():
    if state_key not in all_size_tables:
        continue
    
    size_df = all_size_tables[state_key]
    
    print(f'\n{state_cfg["name"]} — Water Body Changes:')
    print('=' * 60)
    
    for sc_label in size_labels:
        sc_data = size_df[size_df['size_class'] == sc_label]
        if len(sc_data) < 2:
            continue
        
        counts = {row['year']: row['count'] for _, row in sc_data.iterrows()}
        areas = {row['year']: row['total_area_ha'] for _, row in sc_data.iterrows()}
        
        if 2016 in counts and 2025 in counts:
            count_change = counts[2025] - counts[2016]
            area_change = areas[2025] - areas[2016]
            pct_change = (area_change / areas[2016] * 100) if areas[2016] > 0 else 0
            sign = '+' if count_change >= 0 else ''
            print(f'  {sc_label:15s}: {sign}{count_change} bodies, '
                  f'{area_change:+.1f} ha ({pct_change:+.1f}%)')

## 4.5: Visualization

In [ ]:
# Grouped bar charts
for state_key, state_cfg in STATES.items():
    if state_key not in all_size_tables:
        continue
    
    plot_water_size_distribution(all_size_tables[state_key], state_cfg['name'])

print('✅ Bar charts generated.')

In [ ]:
# Map: water bodies colored by size class (side-by-side for 3 years)
size_colors = {
    'Very Small': '#cce5ff',
    'Small': '#66b3ff',
    'Medium': '#3399ff',
    'Large': '#0066cc',
    'Very Large': '#004499',
    'Mega': '#001a66',
}

for state_key, state_cfg in STATES.items():
    available_years = [y for y in YEARS if y in water_bodies.get(state_key, {})]
    if len(available_years) < 2:
        continue
    
    # Load boundary for background
    boundary_file = BOUNDARIES_DIR / f'{state_key}_district_boundaries.geojson'
    boundary_gdf = gpd.read_file(boundary_file) if boundary_file.exists() else None
    
    fig, axes = plt.subplots(1, len(available_years),
                             figsize=(7 * len(available_years), 10))
    if len(available_years) == 1:
        axes = [axes]
    
    for ax, year in zip(axes, available_years):
        gdf = water_bodies[state_key][year]
        
        # Plot boundary
        if boundary_gdf is not None:
            boundary_gdf.plot(ax=ax, color='#f0f0f0', edgecolor='grey', linewidth=0.5)
        
        # Plot water bodies by size class
        for sc_label, color in size_colors.items():
            subset = gdf[gdf['size_class'] == sc_label]
            if len(subset) > 0:
                subset.plot(ax=ax, color=color, markersize=1, alpha=0.8)
        
        ax.set_title(f'{year}', fontsize=13)
        ax.set_axis_off()
    
    # Legend
    legend_patches = [Patch(facecolor=c, label=l) for l, c in size_colors.items()]
    axes[-1].legend(handles=legend_patches, loc='lower right', fontsize=9)
    
    fig.suptitle(f'Water Bodies by Size Class — {state_cfg["name"]}', fontsize=15)
    fig.tight_layout()
    save_figure(fig, f'{state_key}_water_bodies_map.png', 'water_analysis')
    plt.show()

print('\n✅ Module 4 complete.')

---
## Summary
- ✅ Water bodies vectorized at 10m and classified into 6 size classes
- ✅ Summary tables with count and area per class per year
- ✅ Change analysis between periods
- ✅ Grouped bar charts (count + area)
- ✅ Side-by-side spatial maps colored by size class

**Next**: `05_water_buffer.ipynb`